In [ ]:
#1
import pandas as pd
import os
from IPython.display import display, Markdown

import pandas as pd

# Load each CSV
train_df = pd.read_csv('/kaggle/input/datasets/isnatsami/promp-rec/PR_train(Sheet1).csv')
test_df=  pd.read_csv('/kaggle/input/datasets/isnatsami/promp-rec/PR_test(Sheet1).csv')
val_df = pd.read_csv('/kaggle/input/datasets/isnatsami/promp-rec/PR_validation(Sheet1).csv')

# Display first 5 rows of each
print("=== Train Dataset ===")
display(train_df.head())

print("\n=== Test Dataset ===")
display(test_df.head())

print("\n=== Validation Dataset ===")
display(val_df.head())

In [ ]:
#2
import pandas as pd

class_counts = train_df["STYLE"].value_counts()
print(class_counts)
class_counts1 = test_df["STYLE"].value_counts()
print(class_counts1)

class_counts2 = val_df["STYLE"].value_counts()
print(class_counts2)

In [ ]:
#3
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

unique_styles = train_df['STYLE'].unique()
style2id = {style: idx for idx, style in enumerate(unique_styles)}
id2style = {v: k for k, v in style2id.items()}
print(f"\n=== Label Mapping ({len(style2id)} classes) ===")
for style, idx in sorted(style2id.items(), key=lambda x: x[1]):
    count = (train_df['STYLE'] == style).sum()
    print(f"{idx}: {style} ({count} samples)")

In [ ]:
#4
import numpy as np

print("\n=== Checking for NaN values ===")
print(f"Train - ORIGINAL TRANSCRIPTS NaN: {train_df['ORIGINAL TRANSCRIPTS'].isna().sum()}")
print(f"Train - CHANGE STYLE NaN: {train_df['CHANGE STYLE'].isna().sum()}")
print(f"Val - ORIGINAL TRANSCRIPTS NaN: {val_df['ORIGINAL TRANSCRIPTS'].isna().sum()}")
print(f"Val - CHANGE STYLE NaN: {val_df['CHANGE STYLE'].isna().sum()}")
print(f"Test - ORIGINAL TRANSCRIPTS NaN: {test_df['ORIGINAL TRANSCRIPTS'].isna().sum()}")
print(f"Test - CHANGE STYLE NaN: {test_df['CHANGE STYLE'].isna().sum()}")

# ✅ CLEAN THE DATA - Fill NaN with empty strings
train_df['ORIGINAL TRANSCRIPTS'] = train_df['ORIGINAL TRANSCRIPTS'].fillna('').astype(str)
train_df['CHANGE STYLE'] = train_df['CHANGE STYLE'].fillna('').astype(str)

val_df['ORIGINAL TRANSCRIPTS'] = val_df['ORIGINAL TRANSCRIPTS'].fillna('').astype(str)
val_df['CHANGE STYLE'] = val_df['CHANGE STYLE'].fillna('').astype(str)

test_df['ORIGINAL TRANSCRIPTS'] = test_df['ORIGINAL TRANSCRIPTS'].fillna('').astype(str)
test_df['CHANGE STYLE'] = test_df['CHANGE STYLE'].fillna('').astype(str)

print("✅ Data cleaned!")

In [ ]:
#5
print("\n=== Fixing labels ===")

# Remove rows with NaN STYLE
print(f"Train before: {len(train_df)}")
train_df = train_df.dropna(subset=['STYLE']).reset_index(drop=True)
print(f"Train after: {len(train_df)}")

print(f"Val before: {len(val_df)}")
val_df = val_df.dropna(subset=['STYLE']).reset_index(drop=True)
print(f"Val after: {len(val_df)}")

# Create labels
train_df['label'] = train_df['STYLE'].map(style2id)
val_df['label'] = val_df['STYLE'].map(style2id)

# Check for unmapped styles and remove them
if train_df['label'].isna().sum() > 0:
    print(f"⚠️ Dropping {train_df['label'].isna().sum()} train rows with unmapped styles")
    train_df = train_df.dropna(subset=['label']).reset_index(drop=True)
    
if val_df['label'].isna().sum() > 0:
    print(f"⚠️ Dropping {val_df['label'].isna().sum()} val rows with unmapped styles")
    val_df = val_df.dropna(subset=['label']).reset_index(drop=True)

# Convert to int (important!)
train_df['label'] = train_df['label'].astype(int)
val_df['label'] = val_df['label'].astype(int)

print(f"\nFinal train size: {len(train_df)}")
print(f" Final val size: {len(val_df)}")
print(f" Label range: {train_df['label'].min()} to {train_df['label'].max()}")


In [ ]:
#6
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Match the Tokenizer to the Model exactly
model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Load the Model once here
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=9
)
model.to("cuda")

print("✅ Tokenizer and Model are now matched and ready!")

In [ ]:
#8
token_lengths = []

for _, row in train_df.iterrows():
    # Combine them exactly how you do in the Dataset class
    combined_text = row['ORIGINAL TRANSCRIPTS'] + " " + row['CHANGE STYLE']
    tokens = tokenizer.encode(combined_text, add_special_tokens=True)
    token_lengths.append(len(tokens))

# Analyze the results
max_len_found = max(token_lengths)
over_limit = sum(1 for x in token_lengths if x > 384) # Your current limit

print(f"Maximum tokens found: {max_len_found}")
print(f"Samples longer than your current limit (384): {over_limit}")
print(f"Average token length: {sum(token_lengths)/len(token_lengths):.1f}")

In [ ]:
#9
import difflib
import pandas as pd
import torch
from torch.utils.data import Dataset

# --- 1️⃣ STEP 1: UPSAMPLING ---
print("📊 Step 1: Balancing the classes...")
class_counts = train_df['label'].value_counts()
target_count = class_counts.max() 
df_list = []
for label in range(len(style2id)):
    df_class = train_df[train_df['label'] == label]
    if len(df_class) > 0:
        df_class_upsampled = df_class.sample(target_count, replace=True, random_state=42)
        df_list.append(df_class_upsampled)
train_df_balanced = pd.concat(df_list).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"✅ Upsampling done. New size: {len(train_df_balanced)}")

def precompute_sandwich(df, name="Dataset"):
    print(f"⏳ Step 2: Extracting 'Sandwich' text for {name}...")
    results = []
    for idx, row in df.iterrows():
        orig_words = str(row['ORIGINAL TRANSCRIPTS']).split()
        chng_words = str(row['CHANGE STYLE']).split()
        
        s = difflib.SequenceMatcher(None, orig_words[:1000], chng_words[:1000], autojunk=True)
        added_text, context_text = "", ""
        
        for tag, i1, i2, j1, j2 in s.get_opcodes():
            if tag in ('replace', 'insert'):
                added_text = " ".join(chng_words[j1:j2])
                start = max(0, j1 - 20)
                end = min(len(chng_words), j2 + 20)
                context_text = " ".join(chng_words[start:end])
                break 
        if not added_text:
            added_text = "No change"
            context_text = " ".join(chng_words[:50])
            
        results.append(f"మార్పు: {added_text} | సందర్భం: {context_text}")
        if (idx + 1) % 2000 == 0:
            print(f"   Processed {idx + 1}/{len(df)} rows...")
            
    return results

train_df_balanced['sandwich_text'] = precompute_sandwich(train_df_balanced, "Train")
val_df['sandwich_text'] = precompute_sandwich(val_df, "Validation")
test_df['sandwich_text'] = precompute_sandwich(test_df, "Test")

# --- 3️⃣ STEP 3: DATASET & TOKENIZATION (FIXED VERSION) ---
class StyleRecoveryDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        # We store the dataframe directly to use in __getitem__
        self.data = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # We combine the "Sandwich" feature with the full text using [SEP]
        # This gives the model a 'hint' and then the full context
        combined_text = f"{row['sandwich_text']} {self.tokenizer.sep_token} {row['CHANGE STYLE']}"

        encoding = self.tokenizer(
            combined_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        
        # Add label if it exists (Train/Val), else skip (Test)
        if 'label' in row:
            item['labels'] = torch.tensor(int(row['label']), dtype=torch.long)
            
        return item

train_dataset = StyleRecoveryDataset(train_df_balanced, tokenizer)
val_dataset = StyleRecoveryDataset(val_df, tokenizer)
test_dataset = StyleRecoveryDataset(test_df, tokenizer)


In [ ]:
!rm -rf ./muril_results


In [ ]:
!df -h / | awk '{print "Available: " $4}'


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# -- METRICS DEFINITION ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, predictions, average='macro')
    # Adding accuracy here as well so you can report both in your paper
    acc = accuracy_score(labels, predictions)
    return {"macro_f1": macro_f1, "accuracy": acc}

train_df_balanced['label'] = train_df_balanced['label'].astype(int).clip(0, 8)

raw_weights = compute_class_weight('balanced', classes=np.unique(train_df_balanced['label']), y=train_df_balanced['label'])
aligned_weights = [np.sqrt(w) for w in raw_weights] 
device_weights = torch.tensor(aligned_weights, dtype=torch.float).to("cuda")

class FocalStyleTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # 🛡️ THE SEATBELT: This prevents the 'index out of bounds' crash
        safe_labels = labels.clamp(0, 8) 
        
        ce_loss = F.cross_entropy(logits, safe_labels, reduction='none', weight=device_weights)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** 3.0 * ce_loss).mean()
        return (focal_loss, outputs) if return_outputs else focal_loss

# Calculate total steps for warmup_steps (Cleanest way to fix the deprecation warning)
total_steps = (len(train_dataset) // 8) * 15 # (samples // batch_size) * epochs

training_args = TrainingArguments(
    output_dir="./muril_results",
    num_train_epochs=15,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=4e-5,
    lr_scheduler_type="cosine",
    warmup_steps=int(total_steps * 0.1), # Replacing warmup_ratio with explicit steps
    weight_decay=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True,
    logging_steps=1, 
    report_to="none",
    save_total_limit=1,
    dataloader_num_workers=0 
)

# 🛠️ FIXED: Used 'processing_class' instead of 'tokenizer'
trainer = FocalStyleTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer, 
    compute_metrics=compute_metrics
)

print(f"📍 Training on: {torch.cuda.get_device_name(0)}")
print("🚀 Launching training...")
trainer.train()

# Final Evaluation
final_val_results = trainer.evaluate()
print(f" Final Best Validation Macro F1: {final_val_results['eval_macro_f1']:.4f}")

# Generate Submission
test_predictions_output = trainer.predict(test_dataset)
test_pred_indices = np.argmax(test_predictions_output.predictions, axis=1)
final_test_styles = [id2style[idx] for idx in test_pred_indices]

submission_df = pd.DataFrame({"ID": test_df['ID'], "STYLE": final_test_styles})
submission_df.to_csv("predictions.csv", index=False)
print("💾 predictions.csv saved!")

# --- DIAGNOSTIC PLOT ---
predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

style_names = [id2style[i] for i in range(len(style2id))]
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=style_names, yticklabels=style_names)
plt.title('Confusion Matrix: Diagnostic Map')
plt.show()

print(classification_report(y_true, y_pred, target_names=style_names, digits=4))